# Track A: Data and models (US Phillips Curve)
Produces three files for the app: `data/phillips_monthly.csv`, `artifacts/models.joblib`, `artifacts/metrics.json`.
Run top to bottom in Colab. Contract (columns, model keys) is in `src/config.py`; do not rename anything.

In [ ]:
!pip -q install statsmodels
import json, joblib, sklearn, numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
print("sklearn", sklearn.__version__)   # put this exact version in requirements.txt

## 1. Pull the nine FRED series (Review-1 slides 8-10)

In [ ]:
SERIES = {"cpi":"CPIAUCSL","core":"CPILFESL","unrate":"UNRATE","nrou":"NROU","mich":"MICH",
          "t5yie":"T5YIE","fedfunds":"FEDFUNDS","wti":"DCOILWTICO","m2":"M2SL"}
def fred(sid, start="1959-01-01"):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}&cosd={start}"
    s = pd.read_csv(url, index_col=0, parse_dates=True).iloc[:, 0]
    return pd.to_numeric(s, errors="coerce")
raw = {k: fred(v) for k, v in SERIES.items()}
for k, s in raw.items():
    print(f"{k:9s} {SERIES[k]:12s} {s.first_valid_index().date()} -> {s.last_valid_index().date()}  n={s.notna().sum()}")

## 2. Align to monthly and build inflation / M2 growth

In [ ]:
end = raw["cpi"].dropna().index.max()
df = pd.DataFrame(index=pd.date_range("1960-01-01", end, freq="MS"))
df["cpi_yoy"]  = (raw["cpi"].pct_change(12) * 100)
df["core_yoy"] = (raw["core"].pct_change(12) * 100)
df["m2_yoy"]   = (raw["m2"].pct_change(12) * 100)
df["unrate"]   = raw["unrate"]
df["mich"]     = raw["mich"]
df["fedfunds"] = raw["fedfunds"]
df["t5yie"]    = raw["t5yie"].resample("MS").mean()          # daily -> monthly mean
df["wti"]      = raw["wti"].resample("MS").mean()            # daily -> monthly mean
df["nrou"]     = raw["nrou"].resample("MS").ffill()          # quarterly -> monthly (step)
df = df.reindex(pd.date_range("1960-01-01", end, freq="MS"))
df["nrou"] = df["nrou"].ffill()
df.index.name = "date"
COLS = ["cpi_yoy","core_yoy","unrate","nrou","mich","t5yie","fedfunds","wti","m2_yoy"]
df = df[COLS].round(4)
import os; os.makedirs("data", exist_ok=True); os.makedirs("artifacts", exist_ok=True)
df.to_csv("data/phillips_monthly.csv")
print(df.tail(3)); print(df.isna().sum())

## 3. Quick EDA (screenshot these for the slides)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
df[["cpi_yoy","core_yoy"]].plot(ax=ax[0,0], title="Inflation (YoY %)")
df[["unrate","nrou"]].plot(ax=ax[0,1], title="Unemployment vs NROU")
df[["mich","t5yie"]].plot(ax=ax[1,0], title="Expectations: Michigan vs 5y breakeven")
df[["wti"]].plot(ax=ax[1,1], title="WTI oil ($)")
plt.tight_layout(); plt.show()
print(df.loc["2003":].corr().round(2))

## 4. Models on the full-variable sample (2003 onward, all nine series available)
Time-ordered split: **train to 2018-12, test 2019-01 onward**. Metrics come from the train-only fit; the models shipped to the app are refit on all data.
Random Forest hyperparameters are chosen with an expanding-window `TimeSeriesSplit` on the training part only.

In [ ]:
MODELS = {"classic": ["unrate"], "augmented": ["unrate","mich"],
          "full_ols": ["unrate","mich","fedfunds","wti","m2_yoy"], "rf": ["unrate","mich","fedfunds","wti","m2_yoy"]}
ALL = MODELS["rf"]
TARGETS = ["cpi_yoy", "core_yoy"]
d = df.loc["2003-01":].dropna(subset=ALL + TARGETS)
train, test = d.loc[:"2018-12"], d.loc["2019-01":]
print("sample", d.index.min().date(), d.index.max().date(), "n =", len(d), "| train", len(train), "test", len(test))

def tune_rf(X, y):
    gs = GridSearchCV(RandomForestRegressor(n_estimators=300, random_state=42),
                      {"max_depth":[2,3,5], "min_samples_leaf":[5,10,20]},
                      cv=TimeSeriesSplit(5), scoring="neg_root_mean_squared_error")
    return gs.fit(X, y).best_params_

final_models, mmetrics, rf_params = {}, {}, {}
for tgt in TARGETS:
    mmetrics[tgt] = {}
    for key, feats in MODELS.items():
        if key == "rf":
            rf_params[tgt] = tune_rf(train[feats], train[tgt])
            mk = lambda: RandomForestRegressor(n_estimators=300, random_state=42, **rf_params[tgt])
        else:
            mk = LinearRegression
        ev = mk().fit(train[feats], train[tgt]); pred = ev.predict(test[feats])
        fin = mk().fit(d[feats], d[tgt])           # fit on DataFrames so feature names are stored
        final_models[f"{tgt}:{key}"] = fin
        mmetrics[tgt][key] = {"r2_in": float(fin.score(d[feats], d[tgt])),
                              "r2_oos": float(r2_score(test[tgt], pred)),
                              "rmse_oos": float(np.sqrt(mean_squared_error(test[tgt], pred)))}
print("RF params:", rf_params)
print(pd.DataFrame({t:{k:v["r2_oos"] for k,v in m.items()} for t,m in mmetrics.items()}).round(3))

## 5. OLS coefficients with HAC (Newey-West) standard errors

In [ ]:
def hac(y, X, lags=12):
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
coefs = {}
for tgt in TARGETS:
    coefs[tgt] = {}
    for key in ["classic","augmented","full_ols"]:
        r = hac(d[tgt], d[MODELS[key]])
        coefs[tgt][key] = {v: {"coef": float(r.params[v]), "se": float(r.bse[v]), "p": float(r.pvalues[v])} for v in r.params.index}
    print(tgt); print(pd.DataFrame({k: {v: round(x["coef"],3) for v,x in c.items()} for k,c in coefs[tgt].items()}))

## 6. H1: stability across eras (classic and augmented slope)

In [ ]:
ERAS = [("1960s","1960-01","1969-12"), ("1970-83 Stagflation & Volcker","1970-01","1983-12"),
        ("1984-2007 Great Moderation","1984-01","2007-12"), ("2008-19 Post-crisis","2008-01","2019-12"),
        ("2020-today","2020-01",None)]
sub = []
for lab, s, e in ERAS:
    x = df.loc[s:e].dropna(subset=["cpi_yoy","unrate"])
    r = hac(x["cpi_yoy"], x[["unrate"]])
    row = {"era": lab, "n": int(len(x)), "slope_classic": float(r.params["unrate"]), "se_classic": float(r.bse["unrate"])}
    xa = df.loc[s:e].dropna(subset=["cpi_yoy","unrate","mich"])
    if len(xa) >= 36:
        ra = hac(xa["cpi_yoy"], xa[["unrate","mich"]])
        row.update({"slope_aug": float(ra.params["unrate"]), "se_aug": float(ra.bse["unrate"]), "coef_mich": float(ra.params["mich"])})
    sub.append(row)
pd.DataFrame(sub).round(3)

## 7. H2 (NROU drift), H3 (oil), H4 (expectations)

In [ ]:
nr = df["nrou"].dropna()
nrou_stats = {"min": float(nr.min()), "max": float(nr.max()), "latest": float(df["nrou"].loc[:end].iloc[-1])}

def oos_r2(feats, tgt="cpi_yoy"):
    m = LinearRegression().fit(train[feats], train[tgt]); return r2_score(test[tgt], m.predict(test[feats]))
base = oos_r2(["unrate","mich"])
oil = {"r2_oos_gain": float(oos_r2(["unrate","mich","wti"]) - base)}
r = hac(d["cpi_yoy"], d[["unrate","mich","wti"]]); oil.update({"coef_wti": float(r.params["wti"]), "t_wti": float(r.tvalues["wti"])})

r = hac(d["cpi_yoy"], d[["unrate","mich"]])
exp = {"r2_oos_gain": float(base - oos_r2(["unrate"])), "coef_mich": float(r.params["mich"]), "t_mich": float(r.tvalues["mich"])}
# robustness: swap Michigan survey for 5y breakeven (same sample)
s = d.dropna(subset=["t5yie"])
exp["mich_r2_in_same_sample"] = float(LinearRegression().fit(s[["unrate","mich"]], s["cpi_yoy"]).score(s[["unrate","mich"]], s["cpi_yoy"]))
exp["t5yie_swap_r2_in"] = float(LinearRegression().fit(s[["unrate","t5yie"]], s["cpi_yoy"]).score(s[["unrate","t5yie"]], s["cpi_yoy"]))
print(nrou_stats, oil, exp, sep="\n")

## 8. Save artifacts (download these three files and give them to Person B)

In [ ]:
defaults = {"unrate":4.0,"mich":3.0,"fedfunds":2.0,"wti":70.0,"m2_yoy":float(round(d["m2_yoy"].median(),1))}
slider_ranges = {}
for f in ALL:
    lo, hi = float(np.floor(d[f].min())), float(np.ceil(d[f].max()))
    slider_ranges[f] = [lo, hi, float(min(max(defaults[f], lo), hi))]
metrics = {
  "stub": False, "sklearn_version": sklearn.__version__,
  "sample": {"start": str(d.index.min().date()), "end": str(d.index.max().date()), "n": int(len(d))},
  "split": {"train_end": "2018-12", "test_start": "2019-01", "n_train": int(len(train)), "n_test": int(len(test))},
  "train_ranges": {f: [float(d[f].min()), float(d[f].max())] for f in ALL},
  "slider_ranges": slider_ranges, "models": mmetrics, "coefs": coefs, "subperiods": sub,
  "nrou": nrou_stats, "oil": oil, "expectations": exp, "rf_params": rf_params,
}
joblib.dump(final_models, "artifacts/models.joblib")
json.dump(metrics, open("artifacts/metrics.json", "w"), indent=2)
from google.colab import files
for p in ["data/phillips_monthly.csv", "artifacts/models.joblib", "artifacts/metrics.json"]:
    files.download(p)